# EDA Bisnis & Operasional - Reliability PART OMEXP

Laporan ini menjawab pertanyaan operasional: barang apa yang paling sering rusak,
kapan kerusakan terjadi, berapa lama proses perbaikannya, dan pola apa yang perlu
menjadi perhatian tim maintenance. Kerusakan dihitung dari titik awal kerusakan
yang sudah terkonfirmasi; catatan administratif lama (RECON), perpindahan lokasi,
dan proses perbaikan yang masih berjalan tidak otomatis dianggap kerusakan.

Untuk pembahasan kesiapan data dan pemilihan fitur model prediksi, lihat notebook
terpisah `01b_feature_selection_eda.ipynb`.


In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
PROJECT_DIR = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from database import connect
sns.set_theme(style='whitegrid')
def query(sql, params=None):
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params or ())
            return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## 0. Tujuan dan pertanyaan yang dijawab

**Tujuan:** memberi gambaran operasional yang mudah dibaca tentang pola kerusakan
PART, untuk mendukung keputusan maintenance sehari-hari - bukan untuk memilih
fitur model.

Pertanyaan yang dijawab:

1. Barang/model apa yang paling sering rusak?
2. Lokasi dan klien mana yang polanya berbeda?
3. Kapan kerusakan paling sering terjadi?
4. Berapa lama PART bertahan sebelum rusak?
5. Berapa banyak PART yang rusak berulang kali?
6. Setelah rusak, seberapa cepat proses perbaikan selesai dan PART kembali dipasang?
7. Apakah perpindahan lokasi (relokasi) berkaitan dengan kerusakan?
8. Seberapa baik kualitas pencatatan dari tahun ke tahun?

Setiap bagian mengikuti pola: **pertanyaan -> hasil -> apa artinya bagi tim
operasional**.


## 1. Istilah penting (versi sederhana)

| Istilah | Arti sederhana |
|---|---|
| **PART** | Komponen yang dipasang di lokasi/perangkat dan menjadi fokus analisis kerusakan pada laporan ini. |
| **TERMINAL** | Perangkat besar tempat PART dipasang. Dihitung terpisah karena pola kerusakannya berbeda. |
| **Titik mulai rusak** | Waktu saat PART pertama kali dianggap berhenti bekerja karena kerusakan - bukan karena dipindah lokasi atau perawatan rutin. |
| **Masa pakai** | Rentang waktu sejak PART dipasang sampai rusak, dipasang ulang, atau data berhenti tercatat. |
| **RECON** | Catatan administratif lama untuk membetulkan data lokasi di sistem. Bukan kejadian kerusakan atau perpindahan sungguhan, sehingga tidak dipakai menghitung waktu. |
| **Perawatan terjadwal (preventive)** | PART dilepas sesuai jadwal perawatan, bukan karena rusak - kecuali kemudian dikonfirmasi rusak sebelum dipasang lagi. |
| **Perbaikan karena rusak (corrective)** | PART dilepas karena mengalami gangguan/kerusakan. Ini yang dihitung sebagai kerusakan. |
| **Relokasi** | PART dipindah dari satu lokasi ke lokasi lain karena kebutuhan operasional, bukan karena rusak. |
| **Kerusakan berulang** | PART yang mengalami lebih dari satu kali kerusakan sepanjang riwayatnya. |
| **Snapshot 30 hari** | Foto kondisi satu PART pada satu tanggal, dipakai untuk menghitung apakah PART tersebut rusak dalam 30 hari berikutnya. |


## 2. Alur data ringkas

**Pertanyaan:** bagaimana event mentah menjadi dataset observasi yang dapat
diaudit? **Mengapa penting:** kesalahan mapping atau semantic dapat mengubah
urutan waktu dan ground truth.

```text
Data mentah -> cleaning teks -> canonical/fuzzy mapping -> validasi master
            -> semantic event -> operational timeline -> installation cycle
            -> observation dataset 30 hari
```

| Kondisi | Perlakuan pipeline | Keputusan |
|---|---|---|
| RECON | Disimpan untuk audit, tidak ikut durasi operasional | KEEP_AUDIT |
| Tanggal invalid/future | Tidak masuk operational timeline | EXCLUDE_TIME |
| Lokasi fuzzy tidak aman | Nilai mentah disimpan, tidak dipakai sebagai fitur lokasi | REVIEW |
| Model inconsistent | Tidak masuk initial cohort | EXCLUDE_COHORT |
| Failure incomplete flow | Tetap failure, bukan negative | KEEP_POSITIVE_REVIEW |
| Reinstall tanpa failure tercatat | Ditandai unknown, tidak otomatis menjadi negatif | EXCLUDE_NEGATIVE |
| Right-censored dengan coverage aktivitas tidak terkonfirmasi | Tetap tersedia untuk audit dan sensitivity analysis | REVIEW_COVERAGE |

Alias yang disetujui dan singkatan teks disimpan pada tabel mapping `analytics`,
bukan hard-coded di fungsi. Nilai mentah, canonical, metode, dasar mapping, dan
approval tetap dapat ditelusuri.


## 3. Ringkasan data yang dimiliki

**Pertanyaan:** data apa yang tersedia dan berapa besar cakupannya? Metrik utama
adalah jumlah journey, event operasional, unit/model PART, lokasi, cycle, failure,
dan periode. KPI ringkas ditampilkan terlebih dahulu; detail quality dan cohort
dibahas pada chapter berikutnya.


In [ ]:
database_overview = query("""
WITH label_gap AS (
    SELECT c.installation_cycle_id,
        BOOL_OR(o.event_semantic = 'RETURN_FLOW') has_return,
        BOOL_OR(o.event_semantic = 'FAILURE_OUTCOME') has_failure_outcome
    FROM analytics.item_installation_cycle c
    JOIN analytics.item_journey_operational_timeline o
      ON o.item_identifier_clean = c.item_identifier_clean
     AND o.created_on > c.installed_on AND o.created_on <= c.cycle_end_on
    WHERE NOT c.has_observed_failure
      AND o.event_semantic IN ('RETURN_FLOW', 'FAILURE_OUTCOME')
    GROUP BY c.installation_cycle_id
)
SELECT * FROM (
 SELECT 1 urutan, 'Item yang muncul di journal' kelompok, 'Model item yang digunakan' ukuran, (SELECT COUNT(DISTINCT item_model_code_clean) FROM analytics.item_journey_clean)::bigint jumlah, 'Distinct seluruh model pada journey; bukan jumlah unit fisik' keterangan
 UNION ALL SELECT 2, 'Item yang muncul di journal', 'Model PART yang digunakan', (SELECT COUNT(DISTINCT item_model_code_clean) FROM analytics.item_journey_clean WHERE item_category_clean='PART'), 'Model yang benar-benar muncul pada event PART'
 UNION ALL SELECT 3, 'Item yang muncul di journal', 'Model TERMINAL yang digunakan', (SELECT COUNT(DISTINCT item_model_code_clean) FROM analytics.item_journey_clean WHERE item_category_clean='TERMINAL'), 'Model yang benar-benar muncul pada event TERMINAL'
 UNION ALL SELECT 4, 'Item yang muncul di journal', 'Identifier PART yang digunakan', (SELECT COUNT(DISTINCT item_identifier_clean) FROM analytics.item_journey_clean WHERE item_category_clean='PART' AND item_identifier_clean IS NOT NULL), 'Unit PART yang mempunyai sedikitnya satu event'
 UNION ALL SELECT 5, 'Item yang muncul di journal', 'Identifier TERMINAL yang digunakan', (SELECT COUNT(DISTINCT item_identifier_clean) FROM analytics.item_journey_clean WHERE item_category_clean='TERMINAL' AND item_identifier_clean IS NOT NULL), 'Unit TERMINAL yang mempunyai sedikitnya satu event'
 UNION ALL SELECT 6, 'Item yang muncul di journal', 'Lokasi valid yang digunakan', (SELECT COUNT(DISTINCT place_canonical_clean) FROM analytics.item_journey_clean WHERE place_canonical_clean IS NOT NULL), 'Lokasi journey yang berhasil dicocokkan ke master'
 UNION ALL SELECT 7, 'Item yang muncul di journal', 'Klien valid yang digunakan', (SELECT COUNT(DISTINCT client_canonical_clean) FROM analytics.item_journey_clean WHERE client_canonical_clean IS NOT NULL), 'Klien canonical setelah exact/fuzzy mapping aman'
 UNION ALL SELECT 8, 'Aktivitas', 'Seluruh journey/event mentah', (SELECT COUNT(*) FROM analytics.item_journey_clean), 'Termasuk RECON dan event dengan tanggal bermasalah'
 UNION ALL SELECT 9, 'Aktivitas', 'Event operasional', (SELECT COUNT(*) FROM analytics.item_journey_operational_timeline), 'Event yang layak dipakai menyusun urutan waktu'
 UNION ALL SELECT 10, 'Aktivitas', 'Work order', (SELECT COUNT(*) FROM analytics.work_order_clean), 'Dokumen pekerjaan yang tersedia'
 UNION ALL SELECT 11, 'Aktivitas', 'Riwayat status work order', (SELECT COUNT(*) FROM analytics.work_order_history_clean), 'Perubahan status dari seluruh work order'
 UNION ALL SELECT 12, 'Cycle dan failure', 'Event pemasangan', (SELECT COUNT(*) FROM analytics.item_journey_operational_timeline WHERE status_clean='INSTALLED'), 'Satu PART dapat dipasang lebih dari sekali'
 UNION ALL SELECT 13, 'Cycle dan failure', 'Seluruh installation cycle', (SELECT COUNT(*) FROM analytics.item_installation_cycle), 'Cycle sejak pemasangan sampai failure, reinstall, atau cutoff'
 UNION ALL SELECT 14, 'Cycle dan failure', 'Cycle cohort valid', (SELECT COUNT(*) FROM analytics.item_installation_cycle WHERE is_initial_model_cohort), 'Cycle PART yang lolos pemeriksaan awal'
 UNION ALL SELECT 15, 'Cycle dan failure', 'Seluruh failure', (SELECT COUNT(*) FROM analytics.failure_event_clean), 'Corrective dismantle ditambah preventive yang dikonfirmasi rusak'
 UNION ALL SELECT 16, 'Cycle dan failure', 'Failure pada PART', (SELECT COUNT(*) FROM analytics.failure_event_clean WHERE item_category_clean='PART'), 'Failure dengan kategori PART'
 UNION ALL SELECT 17, 'Cycle dan failure', 'Failure pada TERMINAL', (SELECT COUNT(*) FROM analytics.failure_event_clean WHERE item_category_clean='TERMINAL'), 'Dipisahkan dari model failure PART'
 UNION ALL SELECT 18, 'Cycle dan failure', 'Cycle valid yang berakhir failure', (SELECT COUNT(*) FROM analytics.item_installation_cycle WHERE is_initial_model_cohort AND has_observed_failure), 'Failure yang masuk cohort snapshot utama'
 UNION ALL SELECT 19, 'Cycle dan failure', 'Failure dengan alur lanjutan terkonfirmasi', (SELECT COUNT(*) FROM analytics.failure_event_flow WHERE flow_confirmation_status <> 'OPEN_OR_INCOMPLETE_FLOW'), 'Ada RETURN atau proses repair yang cukup sebagai konfirmasi'
 UNION ALL SELECT 20, 'Cycle dan failure', 'Failure tanpa alur lanjutan lengkap', (SELECT COUNT(*) FROM analytics.failure_event_flow WHERE flow_confirmation_status='OPEN_OR_INCOMPLETE_FLOW'), 'Tidak otomatis salah dan tidak menghapus label failure'
 UNION ALL SELECT 21, 'Cycle dan failure', 'Kemungkinan masih berjalan (0-30 hari)', (SELECT COALESCE(SUM(failure_count),0) FROM analytics.eda_incomplete_failure_summary WHERE followup_review_group='LIKELY_ONGOING_0_30D'), 'Terjadi dekat cutoff data'
 UNION ALL SELECT 22, 'Cycle dan failure', 'Perlu dipantau (31-180 hari)', (SELECT COALESCE(SUM(failure_count),0) FROM analytics.eda_incomplete_failure_summary WHERE followup_review_group='REVIEW_31_180D'), 'Belum cukup baru, tetapi belum pasti histori hilang'
 UNION ALL SELECT 23, 'Cycle dan failure', 'Kemungkinan histori hilang (>180 hari)', (SELECT COALESCE(SUM(failure_count),0) FROM analytics.eda_incomplete_failure_summary WHERE followup_review_group='LIKELY_HISTORY_GAP_GT_180D'), 'Prioritas untuk sampling manual'
 UNION ALL SELECT 24, 'Review label', 'Cycle RETURNED tanpa onset failure', (SELECT COUNT(*) FROM label_gap WHERE has_return), 'RETURNED saja belum membuktikan rusak'
 UNION ALL SELECT 25, 'Review label', 'Status rusak jelas tanpa onset tepercaya', (SELECT COUNT(*) FROM label_gap WHERE has_failure_outcome), 'Kandidat review, belum menjadi label utama'
 UNION ALL SELECT 26, 'Dataset model', 'Seluruh snapshot 30 hari', (SELECT COUNT(*) FROM analytics.item_observation_30d), 'Satu PART dapat muncul berkali-kali'
 UNION ALL SELECT 27, 'Dataset model', 'Snapshot layak training', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE is_training_eligible), 'Memiliki label positif atau follow-up negatif penuh'
 UNION ALL SELECT 28, 'Dataset model', 'Snapshot positif', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE is_training_eligible AND target_failure_30d), 'Failure terjadi dalam 30 hari berikutnya'
 UNION ALL SELECT 29, 'Dataset model', 'Snapshot negatif', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE is_training_eligible AND NOT target_failure_30d), 'Tidak failure dan follow-up 30 hari tersedia'
 UNION ALL SELECT 30, 'Dataset model', 'Snapshot dengan follow-up belum lengkap', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE NOT is_training_eligible), 'Dikeluarkan dari training'
 UNION ALL SELECT 31, 'Fitur lokasi', 'Snapshot dengan lokasi master valid', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE is_training_eligible AND is_location_feature_eligible), 'Boleh masuk analisis lokasi'
 UNION ALL SELECT 32, 'Fitur lokasi', 'Snapshot tanpa lokasi master', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE is_training_eligible AND NOT is_location_feature_eligible), 'Tetap diaudit, tetapi lokasi tidak dipakai sebagai fitur'
) ringkasan ORDER BY urutan
""")
database_overview['Jumlah'] = pd.to_numeric(database_overview['jumlah']).map(lambda x: f'{int(x):,}'.replace(',', '.'))
kpi_order = ['Model PART yang digunakan','Identifier PART yang digunakan',
             'Lokasi valid yang digunakan','Seluruh journey/event mentah',
             'Event operasional','Seluruh installation cycle','Seluruh failure',
             'Snapshot layak training']
overview_kpi = database_overview[database_overview.ukuran.isin(kpi_order)].copy()
overview_kpi['urutan_kpi'] = overview_kpi.ukuran.map({name: i for i, name in enumerate(kpi_order)})
overview_kpi = overview_kpi.sort_values('urutan_kpi')
display(overview_kpi[['ukuran','Jumlah','keterangan']].rename(columns={'ukuran':'KPI utama','keterangan':'Cara membaca'}))
modeling_period = query("SELECT MIN(observation_on)::date awal, MAX(observation_on)::date cutoff FROM analytics.item_observation_30d")
display(Markdown(f"**Periode dataset model:** {modeling_period.awal.iloc[0]:%d %B %Y} sampai **cutoff {modeling_period.cutoff.iloc[0]:%d %B %Y}**. Event setelah cutoff atau tanggal yang tidak dipercaya tidak ikut menghitung umur operasional."))


## 4. Sebaran item, model, dan lokasi

**Pertanyaan:** secara umum seperti apa bentuk aktivitas item dan lokasi? Bagian
ini melihat model/lokasi paling aktif, kelompok dengan data sangat sedikit,
status, semantic, dan pola hari kerja sebelum membahas risiko failure.


### 4.1 Distribusi item, model, lokasi, dan hari aktivitas

Jumlah event menunjukkan aktivitas, bukan risiko. Model/lokasi dengan data kecil
ditandai agar persentase ekstrem tidak langsung dipercaya.


In [ ]:
item_activity = query('SELECT * FROM analytics.eda_item_activity_summary ORDER BY event_count DESC')
for col in ['event_count', 'item_count', 'installation_count', 'dismantle_count', 'failure_count']: item_activity[col] = pd.to_numeric(item_activity[col], errors='coerce')
top_items = item_activity.head(15)
rare_items = item_activity.sort_values(['event_count', 'item_count']).head(15)
display(top_items.rename(columns={'item_category_clean': 'Kategori', 'item_model_code_clean': 'Model', 'event_count': 'Event', 'item_count': 'Unit', 'installation_count': 'Installed', 'dismantle_count': 'Dismantled', 'failure_count': 'Failure', 'first_activity_date': 'Pertama', 'last_activity_date': 'Terakhir'}))
display(Markdown(f"Ada **{int(((item_activity.event_count <= 5) | (item_activity.item_count <= 1)).sum())}** model dengan maksimal lima event atau hanya satu unit. Kelompok kecil ini tidak aman dibandingkan berdasarkan persentase tanpa batas minimum sampel."))
display(rare_items[['item_category_clean','item_model_code_clean','event_count','item_count','first_activity_date','last_activity_date']].rename(columns={'item_category_clean': 'Kategori', 'item_model_code_clean': 'Model jarang', 'event_count': 'Event', 'item_count': 'Unit', 'first_activity_date': 'Pertama', 'last_activity_date': 'Terakhir'}))
plt.figure(figsize=(9, 6)); sns.barplot(data=top_items, y='item_model_code_clean', x='event_count', hue='item_category_clean', dodge=False); plt.title('15 model dengan aktivitas operasional terbanyak'); plt.xlabel('Jumlah event'); plt.ylabel('Model'); plt.tight_layout(); plt.show()
location_activity = query('SELECT * FROM analytics.eda_location_activity_summary ORDER BY event_count DESC')
for col in ['event_count', 'item_count', 'model_count', 'installation_count', 'dismantle_count', 'failure_count']: location_activity[col] = pd.to_numeric(location_activity[col], errors='coerce')
top_locations = location_activity.head(15); rare_locations = location_activity.sort_values('event_count').head(15)
display(top_locations.rename(columns={'place_canonical_clean': 'Lokasi', 'event_count': 'Event', 'item_count': 'Unit', 'model_count': 'Model', 'installation_count': 'Installed', 'dismantle_count': 'Dismantled', 'failure_count': 'Failure', 'first_activity_date': 'Pertama', 'last_activity_date': 'Terakhir'}))
display(rare_locations[['place_canonical_clean','event_count','item_count','model_count','first_activity_date','last_activity_date']].rename(columns={'place_canonical_clean': 'Lokasi dengan aktivitas rendah', 'event_count': 'Event', 'item_count': 'Unit', 'model_count': 'Model', 'first_activity_date': 'Pertama', 'last_activity_date': 'Terakhir'}))
plt.figure(figsize=(9, 6)); sns.barplot(data=top_locations, y='place_canonical_clean', x='event_count', color='seagreen'); plt.title('15 lokasi dengan aktivitas operasional terbanyak'); plt.xlabel('Jumlah event'); plt.ylabel('Lokasi'); plt.tight_layout(); plt.show()

display(Markdown(f"""**Temuan utama:** terdapat **{len(item_activity)} kombinasi kategori-model** dan **{len(location_activity)} lokasi canonical** pada event operasional.

**Interpretasi:** volume aktivitas tidak sama dengan failure rate. Analisis risiko berikutnya selalu memakai penyebut snapshot dan minimum support."""))


## 5. Tren operasional dan kerusakan

**Pertanyaan:** bagaimana event, installation, dismantle, failure, dan positive
rate berubah menurut waktu? Perubahan pada 2025 harus dibaca bersama perubahan
detail pencatatan repair, sehingga tren belum otomatis berarti kondisi perangkat
memburuk.


In [ ]:
monthly_trend = query("""SELECT activity_month, SUM(event_count) event_count, SUM(installation_count) installation_count, SUM(dismantle_count) dismantle_count, SUM(failure_count) failure_count FROM analytics.eda_activity_calendar_summary GROUP BY activity_month ORDER BY activity_month""")
monthly_trend['activity_month'] = pd.to_datetime(monthly_trend.activity_month)
trend_columns = ['event_count','installation_count','dismantle_count','failure_count']
monthly_trend[trend_columns] = monthly_trend[trend_columns].apply(pd.to_numeric)
monthly_trend_long = monthly_trend.melt(id_vars='activity_month', value_vars=trend_columns, var_name='metric', value_name='count')
plt.figure(figsize=(13,5)); sns.lineplot(data=monthly_trend_long, x='activity_month', y='count', hue='metric'); plt.title('Tren event operasional per bulan'); plt.xlabel('Bulan'); plt.ylabel('Jumlah'); plt.tight_layout(); plt.show()
yearly = query('SELECT * FROM analytics.eda_failure_rate_by_year ORDER BY observation_year')
yearly['positive_percentage'] = pd.to_numeric(yearly.positive_percentage)
display(yearly.rename(columns={'observation_year':'Tahun','observation_count':'Snapshot','positive_count':'Positif','positive_percentage':'Positive rate (%)'}))
ax=sns.lineplot(data=yearly,x='observation_year',y='positive_percentage',marker='o'); ax.axvline(2025,color='crimson',linestyle='--',label='Era repair detail'); ax.set(title='Positive rate failure 30 hari per tahun',xlabel='Tahun',ylabel='Positive rate (%)'); ax.legend(); plt.show()
rate_2024=float(yearly.loc[yearly.observation_year.eq(2024),'positive_percentage'].iloc[0]); rate_2025=float(yearly.loc[yearly.observation_year.eq(2025),'positive_percentage'].iloc[0]); rate_2026=float(yearly.loc[yearly.observation_year.eq(2026),'positive_percentage'].iloc[0])
display(Markdown(f"""**Temuan utama:** positive rate berubah dari **{rate_2024:.4f}% (2024)** menjadi **{rate_2025:.4f}% (2025)** dan **{rate_2026:.4f}% (2026 parsial)**.

**Interpretasi:** perubahan ini bercampur dengan perubahan pencatatan sejak 2025.

**Keputusan:** validation dan test wajib dipisahkan berdasarkan waktu/era."""))


### 5.1 Tren model dan lokasi operasional

Bagian ini menunjukkan sebaran pemasangan dan perubahan volume, bukan positive
rate. Tujuannya melihat perubahan mix operasional yang dapat menyebabkan drift.


In [ ]:
model_location_scope = query("""WITH scope AS (SELECT item_model_code_clean, COUNT(*) location_count, SUM(installation_count) installations, SUM(item_count) item_location_pairs FROM analytics.eda_item_location_installation_summary GROUP BY item_model_code_clean) SELECT CASE WHEN location_count=1 THEN 'Hanya 1 lokasi' WHEN location_count<=5 THEN '2-5 lokasi' WHEN location_count<=20 THEN '6-20 lokasi' ELSE '>20 lokasi' END location_scope, COUNT(*) model_count, SUM(installations) installations FROM scope GROUP BY 1 ORDER BY MIN(location_count)""")
display(model_location_scope.rename(columns={'location_scope': 'Sebaran model', 'model_count': 'Jumlah model', 'installations': 'Jumlah pemasangan'}))
item_location_matrix = query("""WITH top_model AS (SELECT item_model_code_clean FROM analytics.eda_item_location_installation_summary GROUP BY 1 ORDER BY SUM(installation_count) DESC LIMIT 12), top_location AS (SELECT place_canonical_clean FROM analytics.eda_item_location_installation_summary GROUP BY 1 ORDER BY SUM(installation_count) DESC LIMIT 12) SELECT s.item_model_code_clean, s.place_canonical_clean, s.installation_count FROM analytics.eda_item_location_installation_summary s JOIN top_model m USING (item_model_code_clean) JOIN top_location l USING (place_canonical_clean)""")
item_location_matrix['installation_count'] = pd.to_numeric(item_location_matrix.installation_count); item_location_pivot = item_location_matrix.pivot(index='item_model_code_clean', columns='place_canonical_clean', values='installation_count').fillna(0)
plt.figure(figsize=(13, 8)); sns.heatmap(item_location_pivot, cmap='Blues', annot=True, fmt='.0f'); plt.title('Jumlah pemasangan: 12 model dan 12 lokasi paling aktif'); plt.xlabel('Lokasi'); plt.ylabel('Model PART'); plt.tight_layout(); plt.show()
recent_model_trend = query("""WITH boundary AS (SELECT MAX(created_on) cutoff FROM analytics.item_journey_operational_timeline), recent AS (SELECT DATE_TRUNC('month', o.created_on)::date activity_month, o.item_model_code_clean FROM analytics.item_journey_operational_timeline o CROSS JOIN boundary b WHERE o.item_category_clean='PART' AND o.status_clean='INSTALLED' AND o.created_on>b.cutoff-INTERVAL '36 months'), top_model AS (SELECT item_model_code_clean FROM recent GROUP BY 1 ORDER BY COUNT(*) DESC LIMIT 5) SELECT activity_month, item_model_code_clean, COUNT(*) installation_count FROM recent JOIN top_model USING (item_model_code_clean) GROUP BY 1,2 ORDER BY 1,2""")
recent_model_trend['activity_month'] = pd.to_datetime(recent_model_trend.activity_month); recent_model_trend['installation_count'] = pd.to_numeric(recent_model_trend.installation_count)
plt.figure(figsize=(12, 5)); sns.lineplot(data=recent_model_trend, x='activity_month', y='installation_count', hue='item_model_code_clean', marker='o'); plt.title('Pemasangan lima model teraktif dalam 36 bulan terakhir'); plt.xlabel('Bulan'); plt.ylabel('Jumlah pemasangan'); plt.tight_layout(); plt.show()
recent_location_trend = query("""WITH boundary AS (SELECT MAX(created_on) cutoff FROM analytics.item_journey_operational_timeline), recent AS (SELECT DATE_TRUNC('month', o.created_on)::date activity_month, o.place_canonical_clean FROM analytics.item_journey_operational_timeline o CROSS JOIN boundary b WHERE o.status_clean='INSTALLED' AND o.place_canonical_clean IS NOT NULL AND o.created_on>b.cutoff-INTERVAL '36 months'), top_location AS (SELECT place_canonical_clean FROM recent GROUP BY 1 ORDER BY COUNT(*) DESC LIMIT 8) SELECT activity_month, place_canonical_clean, COUNT(*) installation_count FROM recent JOIN top_location USING (place_canonical_clean) GROUP BY 1,2 ORDER BY 1,2""")
recent_location_trend['installation_count'] = pd.to_numeric(recent_location_trend.installation_count); recent_location_pivot = recent_location_trend.pivot(index='place_canonical_clean', columns='activity_month', values='installation_count').fillna(0)
plt.figure(figsize=(15, 6)); sns.heatmap(recent_location_pivot, cmap='YlOrRd'); plt.title('Kepadatan pemasangan delapan lokasi teraktif dalam 36 bulan terakhir'); plt.xlabel('Bulan'); plt.ylabel('Lokasi'); plt.tight_layout(); plt.show()

## 6. Masa pakai dan pola kerusakan

**Pertanyaan:** berapa lama PART berada dalam cycle sebelum failure/dismantle,
dan cycle mana yang berakhir jelas, reinstall, atau right-censored? Median dan
persentil digunakan karena durasi sangat tidak simetris.


### 6.1 Installation -> kerusakan

Metrik dihitung hanya pada cycle initial cohort dengan failure teramati. Hasil
menjelaskan umur pada saat failure, bukan prediksi umur pasti setiap PART.


In [ ]:
cycle_stats = query("""SELECT COUNT(*) failure_cycles, ROUND(AVG(days_installed_to_failure)::numeric,2) mean_days, ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY days_installed_to_failure)::numeric,2) median_days, ROUND(PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY days_installed_to_failure)::numeric,2) p90_days FROM analytics.item_installation_cycle WHERE is_initial_model_cohort AND has_observed_failure""")
display(cycle_stats.rename(columns={'failure_cycles': 'Siklus dengan kerusakan', 'mean_days': 'Rata-rata hari', 'median_days': 'Median hari', 'p90_days': '90% rusak sebelum hari ke-'}))
durations = query("SELECT LEAST(FLOOR(days_installed_to_failure / 90) * 90, 1800)::int bucket_days, COUNT(*) cycle_count FROM analytics.item_installation_cycle WHERE is_initial_model_cohort AND has_observed_failure GROUP BY 1 ORDER BY 1")
sns.barplot(data=durations, x='bucket_days', y='cycle_count', color='steelblue').set(title='Sebaran umur PART saat mengalami kerusakan', xlabel='Umur sejak dipasang (kelompok 90 hari)', ylabel='Jumlah siklus'); plt.xticks(rotation=45); plt.show()
display(Markdown(f"""**Temuan utama:** median installation-to-failure adalah **{float(cycle_stats.median_days.iloc[0]):,.2f} hari**.

**Tindak lanjut:** bandingkan distribusi menurut model pada baseline; jangan gunakan rata-rata saja.""".replace(',', '.')))


### 6.2 Installation -> dismantle dan lifecycle lokasi

Pasangan lokasi hanya dihitung jika installation dan dismantle dapat dipasangkan.
Cycle tanpa dismantle tidak dipaksa menjadi durasi nol.


In [ ]:
lifecycle_overall = query("""SELECT COUNT(*) installations, COUNT(*) FILTER (WHERE has_next_dismantle) with_next_dismantle, COUNT(*) FILTER (WHERE NOT has_next_dismantle) no_dismantle_before_next_install_or_cutoff, COUNT(*) FILTER (WHERE is_same_location) same_location, COUNT(*) FILTER (WHERE is_location_mismatch) location_mismatch, COUNT(*) FILTER (WHERE has_next_dismantle AND (installed_place_clean IS NULL OR dismantled_place_clean IS NULL)) location_unavailable, ROUND(AVG(days_installed_to_dismantle) FILTER (WHERE is_same_location)::numeric,2) average_days, ROUND(PERCENTILE_CONT(.5) WITHIN GROUP (ORDER BY days_installed_to_dismantle) FILTER (WHERE is_same_location)::numeric,2) median_days, ROUND(PERCENTILE_CONT(.9) WITHIN GROUP (ORDER BY days_installed_to_dismantle) FILTER (WHERE is_same_location)::numeric,2) p90_days, COUNT(*) FILTER (WHERE days_installed_to_dismantle<0) negative_duration FROM analytics.eda_location_lifecycle_detail""")
display(lifecycle_overall.rename(columns={'installations': 'Pemasangan PART', 'with_next_dismantle': 'Memiliki dismantle berikutnya', 'no_dismantle_before_next_install_or_cutoff': 'Belum ada dismantle yang dapat dipasangkan', 'same_location': 'Lokasi installation=dismantle', 'location_mismatch': 'Lokasi berbeda', 'location_unavailable': 'Lokasi tidak tersedia', 'average_days': 'Rata-rata hari', 'median_days': 'Median hari', 'p90_days': 'Persentil 90 hari', 'negative_duration': 'Durasi negatif'}))
lifecycle_location = query("SELECT * FROM analytics.eda_location_lifecycle_summary WHERE matched_lifecycle_count>=20 ORDER BY median_days DESC")
for col in ['installation_count','matched_lifecycle_count','average_days','median_days','p90_days']: lifecycle_location[col] = pd.to_numeric(lifecycle_location[col], errors='coerce')
display(lifecycle_location.rename(columns={'installed_place_clean': 'Lokasi pemasangan', 'installation_count': 'Semua pemasangan', 'matched_lifecycle_count': 'Lifecycle lokasi sama', 'average_days': 'Rata-rata hari', 'median_days': 'Median hari', 'p90_days': 'Persentil 90 hari'}).head(20))
lifecycle_plot = lifecycle_location.sort_values('matched_lifecycle_count', ascending=False).head(15).sort_values('median_days')
plt.figure(figsize=(9, 6)); sns.barplot(data=lifecycle_plot, y='installed_place_clean', x='median_days', color='slateblue'); plt.title('Median hari dari installed sampai dismantle pada lokasi yang sama'); plt.xlabel('Median hari'); plt.ylabel('Lokasi'); plt.tight_layout(); plt.show()
lifecycle_duration = query("SELECT days_installed_to_dismantle FROM analytics.eda_location_lifecycle_detail WHERE is_same_location")['days_installed_to_dismantle'].astype(float)
lifecycle_clip = lifecycle_duration.clip(upper=lifecycle_duration.quantile(.99)); plt.figure(figsize=(9, 4)); sns.histplot(lifecycle_clip, bins=40); plt.title('Distribusi lifecycle lokasi sama (dipotong pada persentil 99 untuk visual)'); plt.xlabel('Hari installed sampai dismantle'); plt.ylabel('Jumlah lifecycle'); plt.tight_layout(); plt.show()
display(Markdown(f"""**Temuan utama:** median lifecycle pada lokasi yang sama adalah **{float(lifecycle_overall.median_days.iloc[0]):,.2f} hari**.

**Keputusan:** gunakan lifecycle lokasi hanya untuk record dengan pasangan waktu dan lokasi valid.""".replace(',', '.')))


## 7. Risiko menurut model, lokasi, dan klien

**Pertanyaan:** model PART, lokasi, dan klien mana yang polanya berbeda? Hasil
ini adalah hubungan (asosiasi), bukan bukti sebab-akibat, dan hanya ditampilkan
untuk kelompok dengan jumlah data yang cukup (minimum support) supaya tidak
menyesatkan.


### 7.1 Menurut model PART

Metrik adalah positive rate pada model dengan minimum 20 PART dan 10 snapshot
positif. Model dengan volume besar tidak otomatis dianggap lebih berisiko.


In [ ]:
by_model = query("""SELECT item_model_code_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 3) positive_pct FROM analytics.item_observation_30d WHERE is_training_eligible GROUP BY 1 HAVING COUNT(DISTINCT item_identifier_clean) >= 20 AND COUNT(*) FILTER (WHERE target_failure_30d) >= 10 ORDER BY positives DESC LIMIT 25""")
display(by_model.rename(columns={'item_model_code_clean': 'Model PART', 'observations': 'Jumlah snapshot', 'items': 'Jumlah PART', 'positives': 'Rusak dalam 30 hari', 'positive_pct': 'Persentase positif'}))
sns.barplot(data=by_model, y='item_model_code_clean', x='positive_pct').set(title='Persentase kerusakan 30 hari menurut model PART', xlabel='Persentase positif (%)', ylabel='Model PART'); plt.show()
display(Markdown(f"""**Temuan utama:** model dengan positive rate tertinggi pada kelompok yang lolos minimum support adalah **{by_model.sort_values('positive_pct',ascending=False).iloc[0].item_model_code_clean}**.

**Interpretasi:** hasil ini adalah asosiasi; umur, era, lokasi, dan mix aktivitas perlu dikontrol pada model."""))


### 7.2 Menurut lokasi dan interaksi model-lokasi

Hanya lokasi canonical yang dipakai. Positive rate dibandingkan pada kelompok
dengan minimum support agar lokasi sibuk atau kombinasi kecil tidak menyesatkan.


In [ ]:
by_location = query("""SELECT last_place_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 4) positive_pct FROM analytics.item_observation_30d WHERE is_training_eligible AND is_location_feature_eligible GROUP BY last_place_clean HAVING COUNT(DISTINCT item_identifier_clean) >= 50 AND COUNT(*) FILTER (WHERE target_failure_30d) >= 10 ORDER BY positive_pct DESC LIMIT 20""")
by_location['positive_pct'] = pd.to_numeric(by_location['positive_pct'], errors='coerce')
display(by_location.rename(columns={'last_place_clean': 'Lokasi terakhir', 'observations': 'Jumlah snapshot', 'items': 'Jumlah PART', 'positives': 'Rusak dalam 30 hari', 'positive_pct': 'Persentase positif'}))
plt.figure(figsize=(9, 7)); sns.barplot(data=by_location, y='last_place_clean', x='positive_pct', color='steelblue').set(title='Persentase kerusakan 30 hari menurut lokasi terakhir', xlabel='Persentase positif (%)', ylabel='Lokasi terakhir'); plt.tight_layout(); plt.show()
model_location = query("""SELECT item_model_code_clean, last_place_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 4) positive_pct FROM analytics.item_observation_30d WHERE is_training_eligible AND is_location_feature_eligible GROUP BY 1, 2 HAVING COUNT(DISTINCT item_identifier_clean) >= 20 AND COUNT(*) FILTER (WHERE target_failure_30d) >= 10 ORDER BY positives DESC""")
model_location['positive_pct'] = pd.to_numeric(model_location['positive_pct'], errors='coerce')
display(model_location.sort_values('positive_pct', ascending=False).head(20).rename(columns={'item_model_code_clean': 'Model PART', 'last_place_clean': 'Lokasi terakhir', 'observations': 'Jumlah snapshot', 'items': 'Jumlah PART', 'positives': 'Rusak dalam 30 hari', 'positive_pct': 'Persentase positif'}))
heatmap_models = model_location.groupby('item_model_code_clean')['positives'].sum().nlargest(10).index
heatmap_locations = model_location.groupby('last_place_clean')['positives'].sum().nlargest(15).index
heatmap_source = model_location[model_location['item_model_code_clean'].isin(heatmap_models) & model_location['last_place_clean'].isin(heatmap_locations)]
heatmap_data = heatmap_source.pivot(index='item_model_code_clean', columns='last_place_clean', values='positive_pct')
plt.figure(figsize=(18, 7)); sns.heatmap(heatmap_data, cmap='YlOrRd', annot=True, fmt='.2f', linewidths=.4, cbar_kws={'label': 'Persentase positif (%)'}); plt.title('Persentase kerusakan menurut kombinasi model dan lokasi'); plt.xlabel('Lokasi terakhir'); plt.ylabel('Model PART'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
display(Markdown(f"""**Temuan utama:** ada **{len(by_location)} lokasi** dan **{len(model_location)} kombinasi model-lokasi** yang lolos minimum support.

**Keputusan:** uji baseline dengan dan tanpa lokasi; jangan menyimpulkan lokasi sebagai penyebab failure dari grafik ini."""))


### 7.3 Menurut klien

**Pertanyaan:** apakah klien tertentu memiliki pola kerusakan yang berbeda?


In [ ]:
by_client = query("""SELECT installed_client_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 3) positive_pct FROM analytics.item_observation_30d WHERE is_training_eligible AND installed_client_clean IS NOT NULL GROUP BY 1 HAVING COUNT(DISTINCT item_identifier_clean) >= 20 AND COUNT(*) FILTER (WHERE target_failure_30d) >= 10 ORDER BY positive_pct DESC""")
display(by_client.rename(columns={'installed_client_clean': 'Klien', 'observations': 'Jumlah snapshot', 'items': 'Jumlah PART', 'positives': 'Rusak dalam 30 hari', 'positive_pct': 'Persentase positif'}))
if len(by_client):
    sns.barplot(data=by_client, y='installed_client_clean', x='positive_pct', color='seagreen').set(title='Persentase kerusakan 30 hari menurut klien', xlabel='Persentase positif (%)', ylabel='Klien'); plt.show()
    display(Markdown(f"""**Temuan utama:** klien dengan persentase kerusakan tertinggi pada kelompok yang lolos minimum support adalah **{by_client.sort_values('positive_pct', ascending=False).iloc[0].installed_client_clean}**.

**Interpretasi:** hasil ini adalah asosiasi, bukan sebab-akibat; klien dapat berkorelasi dengan lokasi atau model PART tertentu."""))
else:
    display(Markdown('**Catatan:** belum ada kombinasi klien dengan data yang cukup (minimum support) untuk dibandingkan.'))

## 8. Kerusakan berulang

**Pertanyaan:** berapa banyak PART yang rusak lebih dari satu kali, dan
seberapa cepat kerusakan berikutnya terjadi setelah kerusakan sebelumnya?


In [ ]:
failure_counts = query("""SELECT failure_count, COUNT(*) item_count FROM (SELECT item_identifier_clean, COUNT(*) failure_count FROM analytics.failure_event_clean WHERE is_initial_model_cohort GROUP BY item_identifier_clean) t GROUP BY failure_count ORDER BY failure_count""")
failure_counts['kelompok'] = failure_counts['failure_count'].apply(lambda n: str(n) if n < 5 else '5 atau lebih')
failure_counts_grouped = failure_counts.groupby('kelompok', as_index=False)['item_count'].sum()
order = ['1', '2', '3', '4', '5 atau lebih']
failure_counts_grouped['kelompok'] = pd.Categorical(failure_counts_grouped['kelompok'], categories=order, ordered=True)
failure_counts_grouped = failure_counts_grouped.sort_values('kelompok')
display(failure_counts_grouped.rename(columns={'kelompok': 'Jumlah kerusakan per PART', 'item_count': 'Jumlah PART'}))
sns.barplot(data=failure_counts_grouped, x='kelompok', y='item_count', color='indianred').set(title='Jumlah PART menurut berapa kali pernah rusak', xlabel='Jumlah kerusakan per PART', ylabel='Jumlah PART'); plt.show()

repeat_gap = query("""WITH failures AS (SELECT item_identifier_clean, failure_onset_on, LAG(failure_onset_on) OVER (PARTITION BY item_identifier_clean ORDER BY failure_onset_on) previous_failure_on FROM analytics.failure_event_clean WHERE is_initial_model_cohort) SELECT COUNT(*) FILTER (WHERE failure_onset_on - previous_failure_on <= INTERVAL '7 days') repeat_7d, COUNT(*) FILTER (WHERE failure_onset_on - previous_failure_on <= INTERVAL '30 days') repeat_30d, COUNT(*) FILTER (WHERE failure_onset_on - previous_failure_on <= INTERVAL '90 days') repeat_90d, COUNT(*) repeat_total FROM failures WHERE previous_failure_on IS NOT NULL""")
display(repeat_gap.rename(columns={'repeat_7d': 'Rusak lagi dalam 7 hari', 'repeat_30d': 'Rusak lagi dalam 30 hari', 'repeat_90d': 'Rusak lagi dalam 90 hari', 'repeat_total': 'Total pasangan kerusakan berurutan'}))
repeat_item_count = int(failure_counts.loc[failure_counts.failure_count > 1, 'item_count'].sum())
display(Markdown(f"""**Temuan utama:** ada **{repeat_item_count:,} PART** yang tercatat rusak lebih dari satu kali. Dari seluruh pasangan kerusakan berurutan pada PART yang sama, **{int(repeat_gap.repeat_30d.iloc[0]):,}** terjadi dalam jarak 30 hari dari kerusakan sebelumnya.

**Tindak lanjut:** PART dengan kerusakan berulang dalam waktu singkat layak diprioritaskan untuk pemeriksaan akar masalah, bukan sekadar diperbaiki berulang.""".replace(',', '.')))

## 9. Efektivitas perbaikan: dari rusak sampai kembali beroperasi

**Pertanyaan:** setelah PART dinyatakan rusak, seberapa sering prosesnya
tercatat lengkap (dikembalikan/diperbaiki), dan berapa lama sampai PART
tersebut dipasang kembali?


In [ ]:
flow_status = query("""SELECT flow_confirmation_status, COUNT(*) failure_count, ROUND(AVG(days_to_return) FILTER (WHERE days_to_return IS NOT NULL)::numeric, 1) avg_days_to_return FROM analytics.failure_event_flow GROUP BY flow_confirmation_status ORDER BY failure_count DESC""")
flow_labels = {'CONFIRMED_RETURN_30D': 'Kembali dalam 30 hari', 'CONFIRMED_RETURN_LATE': 'Kembali, lebih dari 30 hari', 'CONFIRMED_REPAIR_PROCESS_NO_RETURN': 'Proses perbaikan tercatat, belum ada RETURN', 'CONFIRMED_OUTCOME_NO_RETURN': 'Status hasil tercatat, belum ada RETURN', 'OPEN_OR_INCOMPLETE_FLOW': 'Belum ada catatan lanjutan (masih berjalan/tidak lengkap)'}
flow_display = flow_status.assign(flow_confirmation_status=flow_status['flow_confirmation_status'].replace(flow_labels)).rename(columns={'flow_confirmation_status': 'Status lanjutan', 'failure_count': 'Jumlah kerusakan', 'avg_days_to_return': 'Rata-rata hari sampai RETURN'})
display(flow_display)
sns.barplot(data=flow_display, y='Status lanjutan', x='Jumlah kerusakan', color='darkorange').set(title='Status lanjutan setelah kerusakan tercatat', xlabel='Jumlah kerusakan', ylabel=''); plt.show()

return_to_service = query("""WITH failure_cycles AS (SELECT item_identifier_clean, installation_sequence, failure_onset_on FROM analytics.item_installation_cycle WHERE has_observed_failure AND is_initial_model_cohort) SELECT COUNT(*) failure_cycle_count, COUNT(*) FILTER (WHERE next_cycle.installation_sequence IS NOT NULL) reinstalled_count, ROUND(100.0 * COUNT(*) FILTER (WHERE next_cycle.installation_sequence IS NOT NULL) / COUNT(*), 2) reinstalled_pct, ROUND(AVG(EXTRACT(EPOCH FROM (next_cycle.installed_on - fc.failure_onset_on)) / 86400.0) FILTER (WHERE next_cycle.installation_sequence IS NOT NULL)::numeric, 1) avg_days_to_reinstall, ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM (next_cycle.installed_on - fc.failure_onset_on)) / 86400.0) FILTER (WHERE next_cycle.installation_sequence IS NOT NULL)::numeric, 1) median_days_to_reinstall FROM failure_cycles fc LEFT JOIN analytics.item_installation_cycle next_cycle ON next_cycle.item_identifier_clean = fc.item_identifier_clean AND next_cycle.installation_sequence = fc.installation_sequence + 1""")
display(return_to_service.rename(columns={'failure_cycle_count': 'Siklus yang berakhir karena rusak', 'reinstalled_count': 'Kembali dipasang', 'reinstalled_pct': 'Persentase kembali dipasang', 'avg_days_to_reinstall': 'Rata-rata hari sampai dipasang lagi', 'median_days_to_reinstall': 'Median hari sampai dipasang lagi'}))
display(Markdown(f"""**Temuan utama:** dari seluruh siklus yang berakhir karena kerusakan, **{float(return_to_service.reinstalled_pct.iloc[0]):.1f}%** PART tercatat kembali dipasang, dengan median **{float(return_to_service.median_days_to_reinstall.iloc[0]):,.1f} hari** dari waktu rusak sampai dipasang lagi.

**Catatan:** sisanya belum tentu hilang - beberapa masih dalam proses perbaikan atau datanya berhenti sebelum sempat terlihat dipasang lagi (lihat notebook 01b untuk audit kelengkapan flow).""".replace(',', '.')))

## 10. Relokasi dan kaitannya dengan kerusakan

**Pertanyaan:** apakah PART yang lebih sering dipindah lokasi (relokasi)
punya persentase kerusakan yang berbeda?


In [ ]:
relocation_risk = query("""SELECT CASE WHEN prior_relocation_count = 0 THEN '0 (belum pernah dipindah)' WHEN prior_relocation_count = 1 THEN '1 kali' ELSE '2 kali atau lebih' END kelompok_relokasi, COUNT(*) observations, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 3) positive_pct FROM analytics.item_observation_30d WHERE is_training_eligible GROUP BY 1 ORDER BY 1""")
display(relocation_risk.rename(columns={'kelompok_relokasi': 'Riwayat relokasi sebelum snapshot', 'observations': 'Jumlah snapshot', 'positives': 'Rusak dalam 30 hari', 'positive_pct': 'Persentase positif'}))
sns.barplot(data=relocation_risk, x='kelompok_relokasi', y='positive_pct', color='mediumpurple').set(title='Persentase kerusakan 30 hari menurut riwayat relokasi', xlabel='Riwayat relokasi sebelum snapshot', ylabel='Persentase positif (%)'); plt.show()
display(Markdown('**Interpretasi:** ini adalah perbandingan sederhana, bukan bukti sebab-akibat - PART yang sering dipindah bisa jadi memang PART yang lebih tua atau lebih aktif dipakai, sehingga wajar terlihat lebih berisiko. Faktor umur dan aktivitas perlu dikontrol lebih lanjut sebelum menyimpulkan relokasi sebagai penyebab kerusakan.'))

## 11. Kualitas pencatatan dari tahun ke tahun

**Pertanyaan:** seberapa lengkap dan konsisten pencatatan data dari 2013
sampai sekarang?

Sistem pencatatan berubah sekitar tahun 2025: sebelumnya (2013-2024) proses
perbaikan detail belum selalu tercatat rapi; sejak 2025 statusnya lebih
detail (misalnya NEED REPAIR, REPAIRING). Perbandingan di bawah ini bukan
berarti periode lama lebih banyak masalah - lebih ke arah cara mencatatnya
yang berbeda.


In [ ]:
era_summary = query("""SELECT data_era, event_semantic, COUNT(*) event_count FROM analytics.item_journey_semantic GROUP BY data_era, event_semantic ORDER BY data_era, event_count DESC""")
era_labels = {'LEGACY_2013_2024': '2013-2024 (pencatatan lama)', 'DETAILED_REPAIR_2025_PLUS': '2025 dan seterusnya (pencatatan detail)', 'INVALID_DATE': 'Tanggal tidak valid'}
era_display = era_summary.assign(data_era=era_summary['data_era'].replace(era_labels))
era_total = era_display.groupby('data_era', as_index=False)['event_count'].sum().rename(columns={'event_count': 'total'})
era_display = era_display.merge(era_total, on='data_era')
era_display['persentase'] = (100.0 * era_display['event_count'] / era_display['total']).round(2)
top_categories = era_display.groupby('event_semantic')['event_count'].sum().nlargest(8).index
era_plot = era_display[era_display.event_semantic.isin(top_categories)]
plt.figure(figsize=(9, 5)); sns.barplot(data=era_plot, y='event_semantic', x='persentase', hue='data_era'); plt.title('Komposisi jenis kejadian per era pencatatan'); plt.xlabel('Persentase dari kejadian era tersebut (%)'); plt.ylabel(''); plt.tight_layout(); plt.show()
recon_share = era_display.loc[era_display.event_semantic.eq('ADMIN_RECON')]
display(recon_share.rename(columns={'data_era': 'Era pencatatan', 'event_semantic': 'Jenis kejadian', 'event_count': 'Jumlah', 'persentase': 'Persentase dari era ini (%)'})[['Era pencatatan', 'Jumlah', 'Persentase dari era ini (%)']])
display(Markdown('**Catatan:** kejadian RECON (catatan administratif lama) dan tanggal tidak valid sudah dikeluarkan dari perhitungan umur/durasi PART di seluruh laporan ini, sehingga tidak mengotori angka-angka di atas.'))

## 12. Anomali data yang perlu diketahui

**Pertanyaan:** apa saja kejanggalan data yang perlu diketahui pembaca
sebelum memakai angka-angka di laporan ini?

Anomali di bawah ini adalah karakteristik data historis, bukan otomatis
berarti kesalahan - datanya tetap disimpan untuk audit dan tidak dihapus.


In [ ]:
outlier_business = query("SELECT * FROM analytics.eda_outlier_summary ORDER BY affected_count DESC")
outlier_business_labels = {'OPERATIONAL_GAP_GT_10Y': 'Ada jeda pencatatan lebih dari 10 tahun', 'ZERO_OR_NEGATIVE_DURATION_CYCLE': 'Ada masa pakai yang tercatat 0 hari/tidak valid', 'FAILURE_NOT_PRECEDED_BY_INSTALLED': 'Ada kerusakan yang tidak didahului catatan pemasangan', 'ITEM_WITH_5_PLUS_FAILURES': 'Ada PART yang rusak 5 kali atau lebih', 'JOURNEY_MODEL_INCONSISTENT': 'Ada catatan dengan model yang tidak konsisten', 'INVALID_OR_FUTURE_JOURNEY_DATE': 'Ada tanggal yang tidak valid/di masa depan', 'SNAPSHOT_WITHOUT_MASTER_LOCATION': 'Ada snapshot tanpa lokasi resmi yang cocok', 'REINSTALL_WITHOUT_RECORDED_FAILURE': 'Ada PART yang dipasang ulang tanpa catatan kerusakan sebelumnya', 'RIGHT_CENSORED_ACTIVITY_COVERAGE_UNCONFIRMED': 'Ada PART yang riwayatnya berhenti tanpa kepastian (masih dipakai atau tidak)'}
outlier_business_display = outlier_business.assign(check_name=outlier_business['check_name'].replace(outlier_business_labels)).rename(columns={'check_name': 'Kejanggalan', 'affected_count': 'Jumlah', 'explanation': 'Penjelasan singkat'})
display(outlier_business_display)
display(Markdown('**Kenapa ini penting bagi pembaca laporan:** jumlah besar pada salah satu baris di atas bukan berarti ada kesalahan input - misalnya jeda pencatatan panjang bisa berarti sistem lama belum mencatat rapi, bukan berarti PART benar-benar tidak aktif selama itu. Detail teknis dan cara penanganannya ada di notebook `01b_feature_selection_eda.ipynb`.'))

## 13. Kesimpulan dan rekomendasi


In [ ]:
top_model_row = by_model.sort_values('positive_pct', ascending=False).iloc[0] if len(by_model) else None
top_location_row = by_location.sort_values('positive_pct', ascending=False).iloc[0] if len(by_location) else None
display(Markdown(f"""**Kesimpulan operasional**

- Median umur PART sejak dipasang sampai rusak adalah **{float(cycle_stats.median_days.iloc[0]):,.1f} hari**.
- Median lifecycle pada lokasi yang sama (installed sampai dismantle) adalah **{float(lifecycle_overall.median_days.iloc[0]):,.1f} hari**.
- Model PART dengan persentase kerusakan tertinggi (di antara yang datanya cukup): **{top_model_row.item_model_code_clean if top_model_row is not None else '-'}**.
- Lokasi dengan persentase kerusakan tertinggi (di antara yang datanya cukup): **{top_location_row.last_place_clean if top_location_row is not None else '-'}**.
- **{repeat_item_count:,} PART** tercatat pernah rusak lebih dari satu kali.
- Dari siklus yang berakhir karena kerusakan, **{float(return_to_service.reinstalled_pct.iloc[0]):.1f}%** tercatat kembali dipasang, median **{float(return_to_service.median_days_to_reinstall.iloc[0]):,.1f} hari** setelah rusak.
- Lonjakan aktivitas harian terbesar dalam data ini adalah penerimaan gudang massal, bukan kerusakan atau pemasangan - jangan salah baca sebagai lonjakan kerusakan.

**Rekomendasi untuk tim maintenance:**

1. Prioritaskan audit akar masalah pada PART dengan kerusakan berulang dalam waktu singkat (bagian 8), bukan sekadar memperbaiki berulang kali.
2. Gunakan tabel risiko per model dan lokasi (bagian 7) sebagai titik awal diskusi, bukan kesimpulan final - perlu konfirmasi lapangan karena ini baru hubungan statistik.
3. Pantau siklus yang lama tidak kembali dipasang (bagian 9) - dapat menandakan proses perbaikan yang macet atau data yang belum diperbarui.
4. Untuk analisis kesiapan model prediksi dan pemilihan fitur, lihat notebook `01b_feature_selection_eda.ipynb`.""".replace(',', '.')))